In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Olist 데이터 전처리
- 목표: 각 테이블을 개별 전처리한 뒤, 안전하게 조인 가능한 상태로 준비
- 원칙:
  1. 전처리 먼저, 조인은 나중
  2. `orders`를 기준 축(order grain)으로 관리
  3. 다건 테이블(`payments`, `reviews`, `geolocation`)은 집계 후 조인

In [7]:
customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
order_payments = pd.read_csv("olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
cat_tr = pd.read_csv("product_category_name_translation.csv")

## 1) 공통 점검 함수
- 테이블별 행 수, 고유키, 결측치 비율을 빠르게 확인합니다.

In [8]:
def audit_table(df, name, key_cols=None, topn_null=10):
    print(f"\n===== {name} =====")
    print(f"shape: {df.shape}")
    if key_cols:
        for k in key_cols:
            if k in df.columns:
                print(f"unique({k}): {df[k].nunique(dropna=True)} / null: {df[k].isna().sum()}")
    null_rate = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\n[Top null rate %]")
    print(null_rate.head(topn_null))

## 2) orders 전처리 (기준 테이블)
- 문자열 날짜를 datetime으로 변환
- `order_id` 중복 여부 확인

In [9]:
order_dt_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in order_dt_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

audit_table(orders, "orders", key_cols=["order_id", "customer_id"])
print("duplicate order_id:", orders["order_id"].duplicated().sum())
print("\norder_status 분포")
print(orders["order_status"].value_counts(dropna=False))


===== orders =====
shape: (99441, 8)
unique(order_id): 99441 / null: 0
unique(customer_id): 99441 / null: 0

[Top null rate %]
order_delivered_customer_date    2.981668
order_delivered_carrier_date     1.793023
order_approved_at                0.160899
order_id                         0.000000
order_purchase_timestamp         0.000000
order_status                     0.000000
customer_id                      0.000000
order_estimated_delivery_date    0.000000
dtype: float64
duplicate order_id: 0

order_status 분포
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 3) customers 전처리
- 고객 식별자(`customer_id`, `customer_unique_id`) 점검
- 우편번호/지역 결측 확인

In [10]:
audit_table(customers, "customers", key_cols=["customer_id", "customer_unique_id"])
print("duplicate customer_id:", customers["customer_id"].duplicated().sum())


===== customers =====
shape: (99441, 5)
unique(customer_id): 99441 / null: 0
unique(customer_unique_id): 96096 / null: 0

[Top null rate %]
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64
duplicate customer_id: 0


## 4) order_items 전처리
- 주문-상품-셀러 연결 핵심 테이블
- 금액 컬럼(`price`, `freight_value`) 품질 점검

In [11]:
audit_table(order_items, "order_items", key_cols=["order_id", "product_id", "seller_id"])
print("price < 0:", (order_items["price"] < 0).sum())
print("freight_value < 0:", (order_items["freight_value"] < 0).sum())


===== order_items =====
shape: (112650, 7)
unique(order_id): 98666 / null: 0
unique(product_id): 32951 / null: 0
unique(seller_id): 3095 / null: 0

[Top null rate %]
order_id               0.0
order_item_id          0.0
product_id             0.0
seller_id              0.0
shipping_limit_date    0.0
price                  0.0
freight_value          0.0
dtype: float64
price < 0: 0
freight_value < 0: 0


## 5) order_payments 전처리
- 주문당 다건 가능
- 결측/이상치 점검 후 `order_id` 기준 집계 테이블 생성

In [12]:
audit_table(order_payments, "order_payments", key_cols=["order_id"])
print("payment_value < 0:", (order_payments["payment_value"] < 0).sum())
print("payment_installments < 0:", (order_payments["payment_installments"] < 0).sum())

pay_agg = order_payments.groupby("order_id", as_index=False).agg(
    payment_value_total=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_type_nunique=("payment_type", "nunique"),
    payment_sequential_max=("payment_sequential", "max"),
)
audit_table(pay_agg, "pay_agg", key_cols=["order_id"])


===== order_payments =====
shape: (103886, 5)
unique(order_id): 99440 / null: 0

[Top null rate %]
order_id                0.0
payment_sequential      0.0
payment_type            0.0
payment_installments    0.0
payment_value           0.0
dtype: float64
payment_value < 0: 0
payment_installments < 0: 0

===== pay_agg =====
shape: (99440, 5)
unique(order_id): 99440 / null: 0

[Top null rate %]
order_id                    0.0
payment_value_total         0.0
payment_installments_max    0.0
payment_type_nunique        0.0
payment_sequential_max      0.0
dtype: float64


## 6) order_reviews 전처리
- 날짜형 변환
- 리뷰 점수 범위 점검 후 `order_id` 기준 집계

In [13]:
review_dt_cols = ["review_creation_date", "review_answer_timestamp"]
for c in review_dt_cols:
    order_reviews[c] = pd.to_datetime(order_reviews[c], errors="coerce")

audit_table(order_reviews, "order_reviews", key_cols=["review_id", "order_id"])
print("review_score out of 1~5:",
      ((order_reviews["review_score"] < 1) | (order_reviews["review_score"] > 5)).sum())

rev_agg = order_reviews.groupby("order_id", as_index=False).agg(
    review_score_mean=("review_score", "mean"),
    review_count=("review_id", "count"),
    review_creation_min=("review_creation_date", "min"),
)
audit_table(rev_agg, "rev_agg", key_cols=["order_id"])


===== order_reviews =====
shape: (99224, 7)
unique(review_id): 98410 / null: 0
unique(order_id): 98673 / null: 0

[Top null rate %]
review_comment_title       88.341530
review_comment_message     58.702532
review_id                   0.000000
review_score                0.000000
order_id                    0.000000
review_creation_date        0.000000
review_answer_timestamp     0.000000
dtype: float64
review_score out of 1~5: 0

===== rev_agg =====
shape: (98673, 4)
unique(order_id): 98673 / null: 0

[Top null rate %]
order_id               0.0
review_score_mean      0.0
review_count           0.0
review_creation_min    0.0
dtype: float64


## 7) products + category translation 전처리
- 카테고리 영문명 매핑
- 상품 속성 결측치 확인

In [14]:
products = products.merge(cat_tr, on="product_category_name", how="left")
audit_table(products, "products(+translation)", key_cols=["product_id"])


===== products(+translation) =====
shape: (32951, 10)
unique(product_id): 32951 / null: 0

[Top null rate %]
product_category_name_english    1.890686
product_category_name            1.851234
product_photos_qty               1.851234
product_name_lenght              1.851234
product_description_lenght       1.851234
product_weight_g                 0.006070
product_height_cm                0.006070
product_length_cm                0.006070
product_width_cm                 0.006070
product_id                       0.000000
dtype: float64


## 8) sellers 전처리
- 셀러 키/지역 결측 점검

In [15]:
audit_table(sellers, "sellers", key_cols=["seller_id"])
print("duplicate seller_id:", sellers["seller_id"].duplicated().sum())


===== sellers =====
shape: (3095, 4)
unique(seller_id): 3095 / null: 0

[Top null rate %]
seller_id                 0.0
seller_zip_code_prefix    0.0
seller_city               0.0
seller_state              0.0
dtype: float64
duplicate seller_id: 0


## 9) geolocation 전처리 (집계본 생성)
- 원본은 zip prefix 중복이 많아 바로 조인하면 데이터 폭증 위험
- zip prefix 기준 대표값으로 축약해서 사용

In [16]:
audit_table(geolocation, "geolocation", key_cols=["geolocation_zip_code_prefix"])

geo_zip = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first"),
)
audit_table(geo_zip, "geo_zip(aggregated)", key_cols=["geolocation_zip_code_prefix"])


===== geolocation =====
shape: (1000163, 5)
unique(geolocation_zip_code_prefix): 19015 / null: 0

[Top null rate %]
geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64

===== geo_zip(aggregated) =====
shape: (19015, 5)
unique(geolocation_zip_code_prefix): 19015 / null: 0

[Top null rate %]
geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64


## 10) (선택) 조인 직전 준비 완료 체크
- 여기까지 끝나면 테이블 개별 전처리는 완료
- 다음 단계에서 `orders` 중심으로 안전 조인 진행

In [17]:
print("전처리 준비 완료")
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("pay_agg:", pay_agg.shape)
print("rev_agg:", rev_agg.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geo_zip:", geo_zip.shape)

전처리 준비 완료
orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
pay_agg: (99440, 5)
rev_agg: (98673, 4)
products: (32951, 10)
sellers: (3095, 4)
geo_zip: (19015, 5)
